## 1. Imports

In [2]:
# Cell 1: Imports
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
from collections import defaultdict
from scipy.stats import zscore
# Scikit-learn & Survival Analysis
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_ipcw
from sksurv.util import Surv

# Load Data
df = pd.read_csv("./data/X_train/clinical_train.csv")
df_eval = pd.read_csv("./data/X_test/clinical_test.csv")

maf_df = pd.read_csv("./data/X_train/molecular_train.csv")
maf_eval = pd.read_csv("./data/X_test/molecular_test.csv")

target_df = pd.read_csv("./data/target_train.csv")

print("Data Loaded.")

Data Loaded.


## 2. Target Cleaning

In [3]:
# Cell 2: Target Cleaning
target_df.dropna(subset=['OS_YEARS', 'OS_STATUS'], inplace=True)
target_df['OS_YEARS'] = pd.to_numeric(target_df['OS_YEARS'], errors='coerce')
target_df['OS_STATUS'] = target_df['OS_STATUS'].astype(bool)

# Merge target into training data
df = df.merge(target_df[['ID', 'OS_YEARS', 'OS_STATUS']], on='ID', how='inner')

## 3. Parsing functions

In [4]:
# Cell 3: Parsing Functions

def parse_PROTEIN_CHANGE(protein):
    """Extracts the numeric position (hotspot) from protein change string."""
    protein = str(protein)
    if len(protein) == 0 or protein == 'nan':
        return 0
    # Regex to find the number in p.R882H -> 882
    match = re.search(r"(\d+)", protein)
    if match:
        return int(match.group(1))
    return 0

def parse_GENE(gene):
    """Returns the full gene name in uppercase."""
    gene = str(gene)
    if len(gene) == 0 or gene == 'nan':
        return "UNKNOWN"
    return gene.upper()

def parse_CYTO(iscn):
    """Parses ISCN strings for complex karyotypes and translocations."""
    iscn = str(iscn).upper().replace(" ", "")
    results = defaultdict(int)
    
    # 1. Detect Complex Karyotype
    clones = iscn.split("/")
    max_abnormalities = 0
    for clone in clones:
        abnormalities = re.findall(r"([+-]\d+|DEL|ADD|INV|T\(|DER)", clone)
        if len(abnormalities) > max_abnormalities:
            max_abnormalities = len(abnormalities)
    if max_abnormalities >= 3:
        results["Complex_Karyotype"] = 1

    # 2. Specific Translocations/Inversions
    structural = re.findall(r"(T|INV)\((\d+|X|Y)[;]?(\d+|X|Y)?\)", iscn)
    for type_, chrom1, chrom2 in structural:
        chrom2_str = f";{chrom2}" if chrom2 else ""
        key = f"{type_}({chrom1}{chrom2_str})"
        results[key] = 1

    # 3. Numeric changes
    numeric_changes = re.findall(r"(?<![0-9])([+-])(\d+|X|Y)(?=[,/]|$)", iscn)
    for sign, num in numeric_changes:
        key = f"{sign}{num}"
        results[key] = 1

    if not results:
        results["normal"] = 1
    return dict(results)

## 4. Data processing

In [5]:
# Cell 4: Cytogenetics Processing

# Calculate Nmut
if 'Nmut' not in df.columns:
    nmut_train = maf_df.groupby('ID').size().reset_index(name='Nmut')
    nmut_test = maf_eval.groupby('ID').size().reset_index(name='Nmut')
    df = df.merge(nmut_train, on='ID', how='left').fillna({'Nmut': 0})
    df_eval = df_eval.merge(nmut_test, on='ID', how='left').fillna({'Nmut': 0})

total_features = ['Nmut']

# Parse CYTO
for dataset in [df, df_eval]:
    cyto_dicts = dataset['CYTOGENETICS'].apply(parse_CYTO)
    cyto_df = pd.DataFrame(cyto_dicts.tolist(), index=dataset.index).fillna(0)
    
    # Fast merge of columns
    dataset[cyto_df.columns] = cyto_df

# Add only columns relevant to training data
cyto_cols = [c for c in df.columns if any(x in c for x in ['+', '-', 't(', 'inv(', 'Complex', 'normal'])]
total_features.extend(cyto_cols)

/tmp/ipykernel_4899/2730901948.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset[cyto_df.columns] = cyto_df
/tmp/ipykernel_4899/2730901948.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset[cyto_df.columns] = cyto_df
/tmp/ipykernel_4899/2730901948.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newfra

## 5. Molecular features with VAF weighting

In [6]:
# Cell 5: Molecular Features (Optimized)

def process_molecular_features():
    global df, df_eval, total_features
    
    # 1. GENE features (Weighted by Max VAF)
    print("Processing Genes...")
    maf_df['GENE_CLEAN'] = maf_df['GENE'].apply(parse_GENE)
    maf_eval['GENE_CLEAN'] = maf_eval['GENE'].apply(parse_GENE)
    
    gene_pivot_train = maf_df.pivot_table(index='ID', columns='GENE_CLEAN', values='VAF', aggfunc='max', fill_value=0)
    gene_pivot_test = maf_eval.pivot_table(index='ID', columns='GENE_CLEAN', values='VAF', aggfunc='max', fill_value=0)
    
    # Rename columns
    gene_pivot_train.columns = [f"GENE_{c}" for c in gene_pivot_train.columns]
    gene_pivot_test.columns = [f"GENE_{c}" for c in gene_pivot_test.columns]
    
    # Align columns: Ensure test has exact same columns as train
    gene_pivot_test = gene_pivot_test.reindex(columns=gene_pivot_train.columns, fill_value=0)
            
    # Merge
    df = df.merge(gene_pivot_train, on='ID', how='left').fillna(0)
    df_eval = df_eval.merge(gene_pivot_test, on='ID', how='left').fillna(0)
    
    total_features.extend(gene_pivot_train.columns.tolist())
    
    # 2. EFFECT and PROTEIN_CHANGE (One-Hot)
    for feature in ['EFFECT', 'PROTEIN_CHANGE']:
        print(f"Processing {feature}...")
        
        if feature == 'PROTEIN_CHANGE':
            maf_df['temp_feat'] = maf_df[feature].apply(parse_PROTEIN_CHANGE)
            maf_eval['temp_feat'] = maf_eval[feature].apply(parse_PROTEIN_CHANGE)
        else:
            maf_df['temp_feat'] = maf_df[feature]
            maf_eval['temp_feat'] = maf_eval[feature]
            
        dummies_train = pd.get_dummies(maf_df['temp_feat'], prefix=feature)
        dummies_train['ID'] = maf_df['ID']
        agg_train = dummies_train.groupby('ID').max()
        
        dummies_test = pd.get_dummies(maf_eval['temp_feat'], prefix=feature)
        dummies_test['ID'] = maf_eval['ID']
        agg_test = dummies_test.groupby('ID').max()
        
        # Align columns
        agg_test = agg_test.reindex(columns=agg_train.columns, fill_value=0)
        
        # Merge
        df = df.merge(agg_train, on='ID', how='left').fillna(0)
        df_eval = df_eval.merge(agg_test, on='ID', how='left').fillna(0)
        
        total_features.extend(list(agg_train.columns))

process_molecular_features()

Processing Genes...
Processing EFFECT...
Processing PROTEIN_CHANGE...


## 6. Interactions and clinical preparation

In [7]:
# Cell 6: Interactions & Clinical Data

def add_interactions(data):
    if 'GENE_NPM1' in data.columns and 'GENE_FLT3' in data.columns:
        data['INT_NPM1_pos_FLT3_neg'] = ((data['GENE_NPM1'] > 0) & (data['GENE_FLT3'] == 0)).astype(int)
    if 'GENE_TP53' in data.columns and 'Complex_Karyotype' in data.columns:
        data['INT_TP53_Complex'] = ((data['GENE_TP53'] > 0) & (data['Complex_Karyotype'] > 0)).astype(int)
    return data

df = add_interactions(df)
df_eval = add_interactions(df_eval)

# Add interactions if they exist
for feat in ['INT_NPM1_pos_FLT3_neg', 'INT_TP53_Complex']:
    if feat in df.columns:
        total_features.append(feat)

# Clinical Transforms
clinical_cols = ['BM_BLAST', 'WBC', 'ANC', 'PLT', 'HB', 'MONOCYTES']
for col in clinical_cols:
    df[col] = np.log1p(df[col])
    df_eval[col] = np.log1p(df_eval[col])

# Impute
imputer = SimpleImputer(strategy='median')
df[clinical_cols] = imputer.fit_transform(df[clinical_cols])
df_eval[clinical_cols] = imputer.transform(df_eval[clinical_cols])

total_features.extend(clinical_cols)
# Remove duplicates
total_features = list(set([f for f in total_features if f in df.columns]))
print(f"Total Features: {len(total_features)}")

Total Features: 1773


## 7. Cox Ridge grid search

In [8]:
# Cell 7: Cox Grid Search

# Prepare Data
X = df[total_features]
y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', df)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Pipeline
pipe = make_pipeline(
    StandardScaler(),
    CoxnetSurvivalAnalysis(max_iter=100000)
)

# Parameters to tune
param_grid = {
    'coxnetsurvivalanalysis__l1_ratio': [0.1, 0.5, 0.9, 0.95], # 0.95 is close to Lasso (selects features)
    'coxnetsurvivalanalysis__alpha_min_ratio': [0.01, 0.05, 0.1] # Higher values = less overfitting
}

print("Running Grid Search...")
search = GridSearchCV(pipe, param_grid, cv=5, n_jobs=-1, verbose=1)
search.fit(X_train, y_train)

print("Best Params:", search.best_params_)

best_cox = search.best_estimator_
c_index_cox_train = concordance_index_ipcw(y_train, y_train, best_cox.predict(X_train), tau=7)[0]
c_index_cox = concordance_index_ipcw(y_train, y_test, best_cox.predict(X_test), tau=7)[0]
print(f"Cox Train C-Index: {c_index_cox_train:.4f}")
print(f"Cox Test C-Index: {c_index_cox:.4f}")

Running Grid Search...
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best Params: {'coxnetsurvivalanalysis__alpha_min_ratio': 0.1, 'coxnetsurvivalanalysis__l1_ratio': 0.1}
Cox Train C-Index: 0.7438
Cox Test C-Index: 0.6875


## 8. Random Survival Forest

In [9]:
# Cell 8: Random Survival Forest

# RSF doesn't need scaling, but it's fine if we pass raw X_train
print("Training Random Survival Forest...")
rsf = RandomSurvivalForest(
    n_estimators=1000,
    min_samples_split=15,
    min_samples_leaf=10, # Higher leaf size prevents overfitting
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)

rsf.fit(X_train, y_train)

c_index_rsf_train = concordance_index_ipcw(y_train, y_train, rsf.predict(X_train), tau=7)[0]
c_index_rsf = concordance_index_ipcw(y_train, y_test, rsf.predict(X_test), tau=7)[0]
print(f"RSF Train C-Index: {c_index_rsf_train*100:.4f}")
print(f"RSF Test C-Index: {c_index_rsf*100:.4f}")

Training Random Survival Forest...
RSF Train C-Index: 72.7052
RSF Test C-Index: 70.6422


## 9. Evaluating ensemble model

In [10]:
# Evaluate Ensemble on BOTH Train and Test sets
from scipy.stats import zscore

# --- 1. Training Set Evaluation ---
# Get predictions
pred_cox_train = best_cox.predict(X_train)
pred_rsf_train = rsf.predict(X_train)

# Normalize (Z-score) and Average
ensemble_train_pred = (zscore(pred_cox_train) + zscore(pred_rsf_train)) / 2

# Calculate C-Index (Train)
c_index_ens_train = concordance_index_ipcw(y_train, y_train, ensemble_train_pred, tau=7)[0]


# --- 2. Test Set Evaluation ---
# Get predictions
pred_cox_test = best_cox.predict(X_test)
pred_rsf_test = rsf.predict(X_test)

# Normalize (Z-score) and Average
ensemble_test_pred = (zscore(pred_cox_test) + zscore(pred_rsf_test)) / 2

# Calculate C-Index (Test)
c_index_ens_test = concordance_index_ipcw(y_train, y_test, ensemble_test_pred, tau=7)[0]


# --- 3. Print Results ---
print(f"Cox Train C-Index (tau=7): {c_index_cox_train:.4f}")
print(f"Cox Test C-Index (tau=7):  {c_index_cox:.4f}")
print("-"*40)
print(f"RSF Train C-Index (tau=7): {c_index_rsf_train:.4f}")
print(f"RSF Test C-Index (tau=7):  {c_index_rsf:.4f}")
print("-"*40)
print(f"Ensemble Train C-Index (tau=7): {c_index_ens_train:.4f}")
print(f"Ensemble Test C-Index (tau=7):  {c_index_ens_test:.4f}")

Cox Train C-Index (tau=7): 0.7438
Cox Test C-Index (tau=7):  0.6875
----------------------------------------
RSF Train C-Index (tau=7): 0.7271
RSF Test C-Index (tau=7):  0.7064
----------------------------------------
Ensemble Train C-Index (tau=7): 0.7398
Ensemble Test C-Index (tau=7):  0.6980


## 10. Submission retrained on full dataset

### Option 1: Submission based of Random Survival Forest only

In [11]:
# Cell 11.1: Final "Production" Submission (Retraining on ALL data)

# 1. Prepare FULL Training Data (Combine train and local test)
# We use the full 'df' which contains all patients with known outcomes
X_full = df[total_features]
y_full = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', df)

# 2. Retrain the Best Model (RSF) on ALL Data
print("Retraining RSF on full dataset (100% of data)...")
rsf_full = RandomSurvivalForest(
    n_estimators=1000,
    min_samples_split=10,
    min_samples_leaf=20,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)
rsf_full.fit(X_full, y_full)

# 3. Prepare Evaluation Data
# Align columns (fill missing with 0)
X_eval_final = df_eval.reindex(columns=total_features, fill_value=0)

# 4. Predict
final_predictions = rsf_full.predict(X_eval_final)

# 5. Save
submission = pd.DataFrame({
    'ID': df_eval['ID'],
    'risk_score': final_predictions
})

filename = "submission/submission_rsf_full_data.csv"
submission.to_csv(filename, index=False)
print(f"Saved {filename} - Ready for upload!")

Retraining RSF on full dataset (100% of data)...
Saved submission/submission_rsf_full_data.csv - Ready for upload!


### Option 2: Submission based of Cox only

In [12]:
# Cell 11.3: Final Cox Submission (Retraining on ALL data)

# 1. Prepare FULL Training Data
X_full = df[total_features]
y_full = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', df)

# 2. Define the Pipeline with Best Parameters
# We use the parameters found in your GridSearch: l1_ratio=0.1, alpha_min_ratio=0.1
cox_full = make_pipeline(
    StandardScaler(),
    CoxnetSurvivalAnalysis(
        l1_ratio=0.1, 
        alpha_min_ratio=0.1, 
        max_iter=100000
    )
)

# 3. Fit on ALL data
print("Retraining Cox model on full dataset...")
cox_full.fit(X_full, y_full)

# 4. Prepare Evaluation Data
X_eval_final = df_eval.reindex(columns=total_features, fill_value=0)

# 5. Predict and Save
cox_predictions = cox_full.predict(X_eval_final)

submission_cox = pd.DataFrame({
    'ID': df_eval['ID'],
    'risk_score': cox_predictions
})

submission_cox.to_csv("submission/submission_cox_full_data.csv", index=False)
print("Saved submission_cox_full_data.csv")

Retraining Cox model on full dataset...
Saved submission_cox_full_data.csv


### Option 3: Submission averaging Random Survival Forest and Cox model (ie Ensemble model)

In [13]:
# Cell 11.3: Final Ensemble Submission (Cox Full + RSF Full)
from scipy.stats import zscore

# 1. Get predictions from both "Full" models
print("Generating ensemble predictions...")

# Ensure predictions are based on the evaluation set
pred_cox_full = cox_full.predict(X_eval_final)
pred_rsf_full = rsf_full.predict(X_eval_final)

# 2. Normalize (Z-score) and Average
# This puts both models on the same scale (mean=0, std=1) before averaging
ensemble_full_pred = (zscore(pred_cox_full) + zscore(pred_rsf_full)) / 2

# 3. Create Submission DataFrame
submission_ensemble = pd.DataFrame({
    'ID': df_eval['ID'],
    'risk_score': ensemble_full_pred
})

# 4. Save
submission_ensemble.to_csv("submission/submission_ensemble_full_data.csv", index=False)
print("Saved submission_ensemble_full_data.csv")
submission_ensemble.head()

Generating ensemble predictions...
Saved submission_ensemble_full_data.csv


,ID,risk_score
0,KYW1,0.487029
1,KYW2,0.542454
2,KYW3,-0.584742
3,KYW4,0.072804
4,KYW5,0.454739


## 11. Best params for RSF

### 1. Finding params 

In [14]:
# Cell 11.1 : Hyperparameter Tuning for Random Survival Forest
# Run this while waiting for submission limit!

from sklearn.model_selection import GridSearchCV

# 1. Define Model
rsf_tune = RandomSurvivalForest(n_jobs=-1, random_state=42)

# 2. Define Grid
# We test tree depth and leaf size to find the balance between
# learning complex rules (deep trees) and avoiding overfitting (large leaves).
param_grid_rsf = {
    'n_estimators': [1000],          # More trees = more stable
    'min_samples_leaf': [5],     # Higher = less overfitting
    'max_depth': [20],          # None = full growth (risky but powerful)
    'max_features': ['sqrt']      # How many features to check at each split
}

# 3. Run Search
print("Tuning Random Survival Forest (this will take time)...")
search_rsf = GridSearchCV(
    rsf_tune,
    param_grid_rsf,
    cv=3,  # 3-fold is faster than 5-fold
    n_jobs=-1,
    verbose=1
)

search_rsf.fit(X_train, y_train) # Use X_train, y_train (80% split) for tuning

print("Best RSF Params:", search_rsf.best_params_)
print("Best RSF CV Score:", search_rsf.best_score_)

Tuning Random Survival Forest (this will take time)...
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Best RSF Params: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'n_estimators': 1000}
Best RSF CV Score: 0.7235330632445883


Best RSF Params: {'max_depth': 30, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'n_estimators': 1000}
Best RSF CV Score: 0.7250618571112399

### 2. Optimized submission RSF

In [15]:
# Cell 11.2 : Final Optimized Submission (Best RSF Params)

# 1. Define the model with your BEST parameters
rsf_optimized = RandomSurvivalForest(
    n_estimators=1000,
    min_samples_leaf=4,      # Tuned value
    max_depth=None,             # Tuned value
    max_features='sqrt',      # Tuned value
    n_jobs=-1,
    random_state=42
)

# 2. Prepare Full Training Data (combining train + local test)
# Ensure we use all available data for the final push
X_full = df[total_features]
y_full = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', df)

# 3. Fit on ALL data
print("Retraining Optimized RSF on full dataset (100% data)...")
rsf_optimized.fit(X_full, y_full)

# 4. Prepare Evaluation Data
# Ensure columns match exactly
X_eval_final = df_eval.reindex(columns=total_features, fill_value=0)

# 5. Predict
final_predictions = rsf_optimized.predict(X_eval_final)

# 6. Save Submission
submission = pd.DataFrame({
    'ID': df_eval['ID'],
    'risk_score': final_predictions
})
filename = "submission/submission_rsf_optimized.csv"
submission.to_csv(filename, index=False)
print(f"Saved {filename}")

Retraining Optimized RSF on full dataset (100% data)...
Saved submission/submission_rsf_optimized.csv
